# Day 36: Implement the ReAct (Reasoning and Acting) loop manually in Python

## Core Theory (Just-in-Time)

Welcome to Day 36! Today we are diving into the foundation of Agentic AI: the **ReAct** pattern.

### Why ReAct?
Traditional Large Language Models (LLMs) are passive. They take a prompt and generate a response based purely on their internal weights. They suffer from hallucinations, outdated information, and inability to interact with external systems. 

**ReAct (Reasoning and Acting)** shifts the paradigm. Instead of just generating an answer, the LLM is instructed to:
1.  **Reason (Thought):** Analyze the current state, determine what information is missing, and decide what action to take next.
2.  **Act (Action):** Execute a specific tool (e.g., search the web, calculate a number, query a database) with specific parameters.
3.  **Observe (Observation):** Take the result of the tool execution and feed it back into the context.

This cycle (`Thought -> Action -> Observation`) repeats until the LLM determines it has enough information to provide a final answer.

### How it works (The Architecture)
We are building a "while" loop. Inside the loop, we prompt the LLM with the user's question AND a description of the available tools. The LLM's output must follow a strict format (e.g., specifying `Thought: ...`, `Action: ...`, `Action Input: ...`). 

Our Python code parses this output. If it sees an action, our Python code *actually runs the function*, gets the result, appends the observation to the prompt, and loops again. This is "production-first": understanding the raw loop before frameworks like LangGraph abstract it away.
### Security & Production Implications
When implementing Agentic AI in production, security is paramount:
1. **PII Protection**: Agents often act on user data. You must intercept and redact Personally Identifiable Information (PII) before it hits the LLM prompt to prevent data leaks.
2. **Fallbacks**: External APIs and LLMs fail. Your Agent loop must have robust fallback mechanisms, capturing errors and gracefully degrading rather than crashing.
3. **Prompt Injection Guardrails**: Malicious input can trick agents into executing unauthorized tools. Strict parameter validation is required.


## 1. Basic Implementation (Procedural)

Let's start with a foundational, procedural implementation of the ReAct loop. This isolates the core concept (`Thought -> Action -> Observation`) with minimal boilerplate.


In [ ]:
import os
import re
from typing import List, Dict, Callable
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, AIMessage

def basic_calculate(expression: str) -> str:
    """A simple calculator tool."""
    return str(eval(expression, {"__builtins__": {}}, {})) if re.match(r"^[0-9+\-*/. ]+$", expression) else "Error"

BASIC_TOOLS: Dict[str, Callable[[str], str]] = {"calculate": basic_calculate}
BASIC_SYSTEM_PROMPT = """
You are an AI assistant. You can use tools.
Tools:
- calculate: evaluates math expressions

Format strictly:
Question: input
Thought: thought
Action: tool name (e.g., calculate)
Action Input: input string
Observation: result
... repeat until final answer
Thought: I know the answer
Final Answer: answer
"""

def run_basic_react(question: str, max_iterations: int = 3) -> str:
    messages: List[BaseMessage] = [
        SystemMessage(content=BASIC_SYSTEM_PROMPT),
        HumanMessage(content=f"Question: {question}\n")
    ]
    
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.0)

    for i in range(max_iterations):
        try:
            response = llm.invoke(messages).content
        except Exception as e:
            return f"System Error: API call failed - {str(e)}"
            
        messages.append(AIMessage(content=response))
        
        if "Final Answer:" in response:
            return response.split("Final Answer:")[-1].strip()
            
        action_match = re.search(r"Action:\s*(.*)", response)
        input_match = re.search(r"Action Input:\s*(.*)", response)
        
        if action_match and input_match:
            action = action_match.group(1).strip()
            action_input = input_match.group(1).strip()
            
            if action in BASIC_TOOLS:
                observation = BASIC_TOOLS[action](action_input)
            else:
                observation = f"Tool {action} not found."
                
            messages.append(HumanMessage(content=f"Observation: {observation}\n"))
        else:
            messages.append(HumanMessage(content="Error: Invalid format."))
            
    return "Max iterations reached."

# print("Basic Run:", run_basic_react("What is 5 + 5?"))


## 2. Medium Implementation (Clean OOP & State Management)

Now, let's refactor into a cleaner Object-Oriented design. This encapsulates state (message history, registered tools) and behaviors (parsing, execution) into cohesive classes, making the code more maintainable and testable.


In [ ]:
import os

os.environ.setdefault("OPENAI_API_KEY", "sk-dummy-key")
import re
from typing import List, Dict, Callable, Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, AIMessage

class Tool:
    def __init__(self, name: str, func: Callable[[str], str], description: str):
        self.name = name
        self.func = func
        self.description = description
        
    def execute(self, tool_input: str) -> str:
        try:
            return self.func(tool_input)
        except Exception as e:
            return f"Tool execution error: {str(e)}"

class ReActAgent:
    def __init__(self, tools: List[Tool], max_iterations: int = 5):
        self.tools = {tool.name: tool for tool in tools}
        self.max_iterations = max_iterations
        self.llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.0)
        self.system_prompt = self._build_system_prompt()
        
    def _build_system_prompt(self) -> str:
        tool_desc = "\n".join([f"- {t.name}: {t.description}" for t in self.tools.values()])
        return f"""You are an AI assistant.
Tools:
{tool_desc}
Format strictly:
Thought: thought
Action: tool_name
Action Input: input
Observation: result
... repeat until final answer
Thought: I know the answer
Final Answer: answer"""

    def run(self, question: str) -> str:
        messages: List[BaseMessage] = [
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=f"Question: {question}")
        ]
        
        for _ in range(self.max_iterations):
            try:
                response = self.llm.invoke(messages).content
            except Exception as e:
                 return f"System Error: API call failed - {str(e)}"
                 
            messages.append(AIMessage(content=response))
            
            if "Final Answer:" in response:
                return response.split("Final Answer:")[-1].strip()
                
            action_match = re.search(r"Action:\s*(.*)", response)
            input_match = re.search(r"Action Input:\s*(.*)", response)
            
            if action_match and input_match:
                action_name = action_match.group(1).strip()
                action_input = input_match.group(1).strip()
                
                tool = self.tools.get(action_name)
                observation = tool.execute(action_input) if tool else f"Tool {action_name} not found."
                messages.append(HumanMessage(content=f"Observation: {observation}"))
            else:
                messages.append(HumanMessage(content="Error: Invalid format."))
                
        return "Max iterations reached."

def mock_weather(loc: str) -> str:
    return "70F, Sunny" if "tokyo" in loc.lower() else "Unknown"

tools = [Tool("weather", mock_weather, "Gets weather for a location.")]
agent = ReActAgent(tools)
# print("Medium Run:", agent.run("Weather in Tokyo?"))


## 3. Advanced Implementation (Production-Grade, Security, Types)

For production-readiness, we must include strict type hinting, robust error handling, Pydantic for validation, and AI security measures (PII redaction). We also handle exact import syntax required for standard Phase 3 Agent architecture.


In [ ]:
import os

os.environ.setdefault("OPENAI_API_KEY", "sk-dummy-key")
import re
import logging
from typing import List, Dict, Callable, Any, Optional
from pydantic import BaseModel, Field, field_validator, ConfigDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, AIMessage

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class SecureTool(BaseModel):
    """Production-grade Tool representation with metadata validation."""
    name: str = Field(..., description="Tool identifier.")
    description: str = Field(..., description="Instructions for the LLM.")
    func: Callable[[str], str]

    model_config = ConfigDict(arbitrary_types_allowed=True)

    def execute(self, tool_input: str) -> str:
        try:
            logger.info(f"Executing {self.name} with input: {tool_input}")
            return self.func(tool_input)
        except Exception as e:
            logger.error(f"Tool {self.name} failed: {e}")
            return f"System Error executing tool. Please gracefully try another approach."

class PIIRedactor:
    """Intercepts and redacts sensitive information."""
    EMAIL_REGEX = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"
    
    @classmethod
    def redact(cls, text: str) -> str:
        redacted = re.sub(cls.EMAIL_REGEX, "[REDACTED_EMAIL]", text)
        return redacted

class ProductionReActAgent:
    def __init__(self, tools: List[SecureTool], max_iterations: int = 5):
        self.tools = {tool.name: tool for tool in tools}
        self.max_iterations = max_iterations
        self.system_prompt = self._build_system_prompt()
        self.llm = ChatOpenAI(model="gpt-4-turbo", temperature=0.0)
        
    def _build_system_prompt(self) -> str:
        tool_desc = "\n".join([f"- {t.name}: {t.description}" for t in self.tools.values()])
        return f"""You are a secure, production-ready AI agent.
Tools available:
{tool_desc}
Strict Format:
Thought: <reasoning>
Action: <tool_name>
Action Input: <input>
Observation: <result>
... repeat as needed
Thought: I have the final answer
Final Answer: <answer>"""

    def execute(self, user_prompt: str) -> str:
        # 1. PII Redaction on input
        safe_prompt = PIIRedactor.redact(user_prompt)
        logger.info(f"Processing sanitized prompt: {safe_prompt}")
        
        messages: List[BaseMessage] = [
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=f"Question: {safe_prompt}")
        ]
        
        for iteration in range(self.max_iterations):
            try:
                response = self.llm.invoke(messages).content
            except Exception as e:
                logger.error(f"LLM Call failed: {e}")
                return "Final Answer: I am currently experiencing system degradation. Please try again later."
                
            messages.append(AIMessage(content=response))
            
            if "Final Answer:" in response:
                return response.split("Final Answer:")[-1].strip()
                
            action_match = re.search(r"Action:\s*(.*)", response)
            input_match = re.search(r"Action Input:\s*(.*)", response)
            
            if action_match and input_match:
                action_name = action_match.group(1).strip()
                action_input = input_match.group(1).strip()
                
                tool = self.tools.get(action_name)
                if tool:
                     observation = tool.execute(action_input)
                else:
                     observation = f"Tool {action_name} not found."
                     
                messages.append(HumanMessage(content=f"Observation: {observation}"))
            else:
                messages.append(HumanMessage(content="Error: Invalid format."))
                
        return "Error: Max iterations reached without finding a final answer."

def mock_user_lookup(email: str) -> str:
    return f"User data for {email}: Active User"

prod_tools = [SecureTool(name="user_lookup", func=mock_user_lookup, description="Looks up user data by email.")]
prod_agent = ProductionReActAgent(prod_tools)
# print("Advanced Run:", prod_agent.execute("Check account for john.doe@example.com"))


## Common Pitfalls in Production

1.  **Infinite Loops:** If the LLM gets confused or a tool repeatedly fails, the agent might loop forever. *Always* implement a `max_iterations` counter to forcefully terminate the loop.
2.  **Parsing Failures:** We relied on regex parsing strings (`Action: ...`). Weaker LLMs or overly complex prompts often cause the LLM to deviate from this strict text format, breaking the parser. Modern solutions (like OpenAI's Tool Calling / JSON mode) solve this by enforcing structured JSON outputs.
3.  **Context Window Exhaustion:** Every iteration appends the Thought, Action, and Observation to the `messages` list. If a tool returns a massive payload (e.g., full HTML of a webpage), you will quickly blow past the LLM's context token limit. Production agents need mechanisms to summarize or truncate large observations.
4.  **Tool Hallucination:** The LLM might try to call `Action: get_stock_price` even if we only gave it `calculate` and `get_weather`. Your code must gracefully handle `action_name not in TOOLS`.

## Common Pitfalls in Production

1. **Infinite Loops:** The ReAct pattern is prone to looping infinitely if a tool fails repeatedly or the agent gets stuck in a thought cycle. **Always** implement a strict `max_iterations` cutoff.
2. **Context Window Exhaustion:** Every iteration appends the Thought, Action, and Observation to the `messages` list. If a tool returns a massive payload (e.g., a full HTML dump), you will quickly blow past the token limit. You must truncate or summarize large observations.
3. **Parsing Failures:** We relied on regex parsing strings (`Action: ...`). Weaker LLMs or overly complex prompts often cause the LLM to deviate from this text format, breaking the parser. Modern agents use Tool Calling (JSON mode) to enforce structured outputs.
4. **Tool Hallucination:** The LLM might try to call `Action: get_stock_price` even if we only provided `calculate` and `weather`. Your code must gracefully handle `action_name not in TOOLS` to prevent exceptions.


## Practical Lab / Homework

**Your Task:**
1. Using the `ProductionReActAgent` class from the Advanced section, create a new tool called `hash_data` that takes a string and returns its SHA-256 hash (use Python's built-in `hashlib`).
2. Register the tool as a `SecureTool`.
3. Instantiate the agent and run it with a prompt instructing it to hash a specific piece of redacted data.
4. **Deliverable:** Record a brief 2-minute async video walkthrough explaining your design decisions regarding state management and how the `SecureTool` Pydantic model ensures type safety in your implementation.


In [ ]:
import hashlib

# Lab Implementation Space
def hash_data(data: str) -> str:
    """Returns the SHA-256 hash of the input string."""
    return hashlib.sha256(data.encode()).hexdigest()

lab_tools = [
    SecureTool(
        name="hash_data",
        func=hash_data,
        description="Hashes input data using SHA-256."
    )
]

lab_agent = ProductionReActAgent(tools=lab_tools)
# lab_result = lab_agent.execute("Please hash the word 'engineering'")
# print(f"Lab Result: {lab_result}")


## Reference Links
- [ReAct Paper (Yao et al., 2022)](https://arxiv.org/abs/2210.03629)
- [LangChain Core Messages Documentation](https://python.langchain.com/v0.2/docs/concepts/#messages)
- [Pydantic v2 Documentation](https://docs.pydantic.dev/latest/)
